# scikit-learn Masterclass — Regression, XGBoost, and Neural Nets
### Modelling & Advanced Data Analysis · 90-minute live walkthrough

**Learning objectives.** By the end you can:
1. Explain the **estimator API** (`fit` / `predict` / `transform` / `predict_proba`) that *every* sklearn model shares.
2. Do a correct **train/test split** and say *why* it exists.
3. Prepare **numeric, categorical, and missing** data with `ColumnTransformer` + `Pipeline`.
4. Train a **regression** model, an **XGBoost classifier**, and a **neural-net classifier**.
5. Recognise and prevent **data leakage** — and explain why it is dangerous.
6. **Save** a model and **deploy** it (Streamlit / API), the bridge to your final project.

> **Instructor pacing (≈90 min).** §1–2 setup & split (10) · §3 regression (12) · §4 data prep (15) · §5 XGBoost (15) · §6 **leakage** (15) · §7 NN (8) · §8 tuning (5) · §9–10 save & deploy (10). If short on time, the leakage section (§6) is the one not to cut — it is the part students get wrong in projects *and* exams.

*All datasets here ship **inside** scikit-learn or are generated in-memory, so the notebook runs offline with no downloads.*

In [ ]:
# If a package is missing, uncomment:
# %pip install scikit-learn xgboost pandas matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, xgboost
print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)
rng = np.random.default_rng(42)   # one reproducible random source for the whole notebook

## 1 · The one mental model behind all of sklearn

Every model ("estimator") in scikit-learn exposes the **same handful of methods**. Learn them once, reuse them everywhere — that consistency is the entire point of the library.

| Method | What it does | Used on |
|---|---|---|
| `.fit(X, y)` | **Learns** parameters from training data | every estimator |
| `.predict(X)` | Produces predictions for new rows | models |
| `.predict_proba(X)` | Class **probabilities** (not just the label) | classifiers |
| `.transform(X)` | Outputs transformed features | preprocessors (scaler, encoder…) |
| `.fit_transform(X)` | `fit` then `transform` in one call | preprocessors |

**Conventions:** `X` is a 2-D array/DataFrame `(n_samples, n_features)`; `y` is the 1-D target. The same `LinearRegression().fit(X, y)` shape works for an XGBoost model or a neural net — only the import changes.

## 2 · The golden rule: split before you do *anything* else

We fit on **training** data and judge on a held-out **test** set the model has never seen. A model that memorises the training rows but fails on new ones has **overfit** — it's useless in production. The test set is our honest estimate of *generalisation*.

- `test_size=0.2` → 80/20 split.
- `random_state=42` → reproducible split (same rows every run).
- `stratify=y` → keep the class proportions identical in train and test (essential for classification, especially when classes are imbalanced).

> **Mantra for the exam:** *the test set is touched exactly once, at the very end.* Any decision (scaling, feature selection, tuning) made using test rows corrupts your estimate.

In [ ]:
from sklearn.model_selection import train_test_split

# toy example just to see the shapes
X_demo = pd.DataFrame({"a":[1,2,3,4,5,6,7,8,9,10], "b":range(10,20)})
y_demo = pd.Series([0,0,0,0,0,1,1,1,1,1])
Xtr, Xte, ytr, yte = train_test_split(X_demo, y_demo, test_size=0.3,
                                       random_state=42, stratify=y_demo)
print("train rows:", len(Xtr), "| test rows:", len(Xte))
print("train class balance:\n", ytr.value_counts(normalize=True).round(2))

## 3 · Regression — predicting a number

**Dataset:** the built-in *diabetes* set — 10 numeric features, target = disease progression after one year.

Workflow: load → split → scale → fit → predict → evaluate.

In [ ]:
from sklearn.datasets import load_diabetes
dia = load_diabetes(as_frame=True, scaled=False)
X, y = dia.data, dia.target
X.head()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# A Pipeline chains steps. StandardScaler is fit on TRAIN ONLY (see §6 for why).
reg = Pipeline([("scale", StandardScaler()),
                ("model", LinearRegression())])
reg.fit(Xtr, ytr)                 # <-- the universal .fit
pred = reg.predict(Xte)           # <-- the universal .predict

rmse = mean_squared_error(yte, pred) ** 0.5
print(f"R²   = {r2_score(yte, pred):.3f}   (1.0 = perfect, 0 = no better than the mean)")
print(f"RMSE = {rmse:.1f}   (avg error, same units as target)")
print(f"MAE  = {mean_absolute_error(yte, pred):.1f}")

In [ ]:
# Swap the final estimator — identical API, no other change. That's the whole lesson.
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=300, random_state=42).fit(Xtr, ytr)
print(f"RandomForest R² = {r2_score(yte, rf.predict(Xte)):.3f}")

**Plot predicted vs actual** — points should hug the diagonal.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(yte, pred, alpha=0.6)
lims = [yte.min(), yte.max()]
plt.plot(lims, lims, "r--", label="perfect")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.legend(); plt.title("Linear regression")
plt.tight_layout(); plt.show()

> 🧑‍🎓 **Mini-exercise.** Replace `LinearRegression` with `Ridge(alpha=1.0)` (from `sklearn.linear_model`). Does R² change? Try `alpha=100`. What is `alpha` doing?

## 4 · Preparing real-world data: numbers + categories + missing values

Real tables are messy. You **cannot** feed text categories or `NaN`s straight into a model. We build a synthetic *customer-churn* table that has all three problems, then fix them with one reusable object.

The tools:
- `SimpleImputer` — fill missing values.
- `StandardScaler` — put numeric features on a comparable scale.
- `OneHotEncoder` — turn categories into 0/1 columns (`handle_unknown="ignore"` protects you from unseen categories at predict time).
- `ColumnTransformer` — apply *different* preprocessing to *different* columns.

In [ ]:
n = 1500
df = pd.DataFrame({
    "age":          rng.integers(18, 70, n).astype(float),
    "income":       rng.normal(50000, 15000, n).round(0),
    "credit_score": rng.normal(650, 80, n).round(0),
    "region":       rng.choice(["north","south","east","west"], n),
    "plan":         rng.choice(["basic","plus","premium"], n, p=[.5,.3,.2]),
    "is_member":    rng.choice([0,1], n),
})
df.loc[rng.random(n) < 0.08, "income"] = np.nan          # inject missing values
logit = (-2 + 0.00002*(50000-df.income.fillna(50000)) + 0.4*(df.plan=="basic")
         - 0.01*(df.credit_score-650) + rng.normal(0,1,n))
df["churn"] = (1/(1+np.exp(-logit)) > 0.5).astype(int)    # the target
print("missing income:", int(df.income.isna().sum()), "| churn rate:", round(df.churn.mean(),3))
df.head()

**⚠️ Class imbalance.** Only ~9% of customers churn. A lazy model that always predicts "no churn" already scores ~91% accuracy. So **accuracy alone is misleading** — we'll lean on ROC-AUC, precision and recall instead (§5).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

num_cols  = ["age", "income", "credit_score"]
cat_cols  = ["region", "plan"]
pass_cols = ["is_member"]                       # already 0/1, leave as-is

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale",  StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("pass", "passthrough", pass_cols),
])

X, y = df.drop(columns="churn"), df["churn"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Peek at what the data looks like AFTER preprocessing
out = preprocess.fit_transform(Xtr)
print("shape after one-hot + scaling:", out.shape, "(extra cols are the one-hot categories)")

## 5 · Classification with XGBoost

XGBoost is gradient-boosted decision trees — the workhorse for tabular data and a near-default for competitions. Same `fit`/`predict` API, plus `predict_proba` for probabilities.

We start on the clean built-in *breast-cancer* set (30 numeric features) so the metrics are easy to read, then drop XGBoost into the messy churn pipeline from §4.

In [ ]:
from sklearn.datasets import load_breast_cancer
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, RocCurveDisplay)

bc = load_breast_cancer(as_frame=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

xgb = XGBClassifier(
    n_estimators=300,      # number of trees
    max_depth=4,           # tree depth — bigger = more complex = more overfit risk
    learning_rate=0.05,    # how much each tree contributes
    subsample=0.9,         # row sampling → regularisation
    eval_metric="logloss",
    random_state=42,
)
xgb.fit(Xb_tr, yb_tr)

y_pred  = xgb.predict(Xb_te)
y_proba = xgb.predict_proba(Xb_te)[:, 1]     # P(class = 1)
print(f"Accuracy = {accuracy_score(yb_te, y_pred):.3f}")
print(f"ROC-AUC  = {roc_auc_score(yb_te, y_proba):.3f}")
print("\n", classification_report(yb_te, y_pred, target_names=bc.target_names))

In [ ]:
# Confusion matrix + ROC curve
fig, ax = plt.subplots(1, 2, figsize=(10,4))
cm = confusion_matrix(yb_te, y_pred)
ax[0].imshow(cm, cmap="Blues")
for (i,j),v in np.ndenumerate(cm): ax[0].text(j,i,v,ha="center",va="center",fontsize=14)
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(yb_te, y_proba, ax=ax[1]); ax[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

In [ ]:
# Which features drive the model?
imp = pd.Series(xgb.feature_importances_, index=bc.data.columns).sort_values().tail(10)
imp.plot.barh(figsize=(6,4), title="Top-10 feature importances"); plt.tight_layout(); plt.show()

In [ ]:
# XGBoost on the MESSY churn data — preprocessing + model as ONE object.
churn_model = Pipeline([("prep", preprocess),
                        ("clf", XGBClassifier(n_estimators=200, max_depth=3,
                                              learning_rate=0.1, eval_metric="logloss",
                                              random_state=42))])
churn_model.fit(Xtr, ytr)
proba = churn_model.predict_proba(Xte)[:, 1]
print(f"Churn  accuracy = {accuracy_score(yte, churn_model.predict(Xte)):.3f}  "
      f"(beware: all-zero baseline ≈ {1-yte.mean():.3f})")
print(f"Churn  ROC-AUC  = {roc_auc_score(yte, proba):.3f}   <-- the metric that actually tells us something")

> 🧑‍🎓 **Mini-exercise.** XGBoost has `scale_pos_weight` to counter imbalance. Set it to `(#negatives / #positives)` and re-fit `churn_model`'s classifier. Watch recall on the churn class move.

## 6 · Data leakage — why your "great" model fails in production

**Data leakage** = information from outside the training fold sneaks into training, so your test score is *too optimistic*. You ship a model that looked excellent offline and then disappoints on live data — and you can't trust *any* of your numbers. Three common forms:

1. **Train/test contamination** — fitting a scaler / imputer / feature-selector on the *full* dataset before splitting.
2. **Target leakage** — a feature that secretly encodes the answer (e.g. `was_refunded` when predicting `will_purchase`).
3. **Temporal leakage** — using future information to predict the past.

### Live demonstration: leakage invents signal out of *pure noise*
We make 1000 totally random features and a random target — there is **no real relationship**. Honest accuracy must be ~50%. Watch what "selecting the best features on the whole dataset" does to that number.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier

n_obs, n_feat = 200, 1000
X_noise = rng.normal(size=(n_obs, n_feat))
y_noise = rng.integers(0, 2, n_obs)          # target is independent of every feature

clf = MLPClassifier(hidden_layer_sizes=(20,), max_iter=300, random_state=0)

# ❌ WRONG — pick the 20 "best" features using ALL the data, THEN cross-validate
X_peeked = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(clf, X_peeked, y_noise, cv=5).mean()

# ✅ RIGHT — selection lives INSIDE the pipeline, re-fit on each train fold only
honest_pipe = Pipeline([("select", SelectKBest(f_classif, k=20)), ("clf", clf)])
honest = cross_val_score(honest_pipe, X_noise, y_noise, cv=5).mean()

print(f"❌ leaky  CV accuracy = {leaky:.3f}   <-- looks like we found signal!")
print(f"✅ honest CV accuracy = {honest:.3f}   <-- the truth: it's just noise (~0.5)")

The leaky number is dangerously inflated even though **the data is random**. The selector peeked at the test rows when choosing features.

**The fix is structural, not a matter of being careful:** put every data-dependent step (imputer, scaler, encoder, selector) *inside* a `Pipeline`. Then `cross_val_score` / `train_test_split` re-fit those steps on each training fold only — leakage becomes impossible by construction. This is the real reason we've used `Pipeline` since §3.

## 7 · A neural network as a classifier (`MLPClassifier`)

`MLPClassifier` is sklearn's built-in feed-forward neural net — **same API again**. Two things matter for NNs specifically:
- **Scaling is not optional.** Gradient descent struggles when features span wildly different ranges. We prove it below.
- `hidden_layer_sizes=(32, 16)` = two hidden layers of 32 and 16 neurons.

For serious deep learning you'd move to Keras/PyTorch, but the `fit`/`predict` concepts carry straight over.

In [ ]:
# Same net, with and without scaling
nn_raw = MLPClassifier(hidden_layer_sizes=(32,16), max_iter=500, random_state=42)
nn_raw.fit(Xb_tr, yb_tr)

nn_scaled = Pipeline([("scale", StandardScaler()),
                      ("nn", MLPClassifier(hidden_layer_sizes=(32,16), max_iter=500, random_state=42))])
nn_scaled.fit(Xb_tr, yb_tr)

print(f"NN without scaling: {accuracy_score(yb_te, nn_raw.predict(Xb_te)):.3f}")
print(f"NN with scaling:    {accuracy_score(yb_te, nn_scaled.predict(Xb_te)):.3f}   <-- same model, just scaled inputs")

## 8 · Doing it right: cross-validation + hyperparameter tuning

A single split is noisy. **k-fold cross-validation** rotates the held-out fold k times and averages — a more stable estimate. `GridSearchCV` wraps this to search hyperparameters, and because we pass the *whole pipeline*, preprocessing is re-fit inside every fold (no leakage during tuning either).

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    churn_model,                                   # the full pipeline from §5
    param_grid={"clf__max_depth": [2, 3, 4],       # note the 'step__param' naming
                "clf__n_estimators": [100, 200]},
    cv=3, scoring="roc_auc", n_jobs=-1)
search.fit(Xtr, ytr)
print("best ROC-AUC:", round(search.best_score_, 3))
print("best params :", search.best_params_)
best_model = search.best_estimator_

## 9 · Save the trained model — the handoff to deployment

`pickle` serialises the **entire pipeline** (preprocessing + model) to one file. At predict time you load it and feed it *raw* data in the original column format — the pipeline redoes all the preprocessing identically. This is exactly what your final-project backend will load.

In [ ]:
import pickle

with open("churn_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# ... later, in a totally separate process / server ...
with open("churn_model.pkl", "rb") as f:
    loaded = pickle.load(f)

new_customer = pd.DataFrame([{
    "age": 41, "income": 38000, "credit_score": 600,
    "region": "south", "plan": "basic", "is_member": 0,
}])
print("prediction:", loaded.predict(new_customer)[0],
      "| churn probability:", round(loaded.predict_proba(new_customer)[0,1], 3))

## 10 · Deploying it — your final project

Two architectures, both valid for the project:

**A · Train once, deploy the artefact (recommended default).**
`notebook → pickle.dump(model) → backend loads it → predicts on the fly.`
Fast responses, cheap, reproducible. The model file is just another asset you ship.

**B · Train on demand.** A button fetches fresh data from an API, trains, then predicts. More dynamic, but slower per request and harder to reproduce — only choose it if your project *needs* fresh-fit models.

### Minimal Streamlit backend (pattern A)
A `app.py` is included alongside this notebook — it loads `churn_model.pkl` and serves predictions from form inputs. Run it with:
```bash
streamlit run app.py
```

### React + Vercel front-end
Same model, different shell: wrap the loaded pipeline in a small **FastAPI** (or Flask) endpoint that accepts JSON and returns `predict_proba`; deploy it as a serverless function, and have your React app `fetch()` it. The browser never runs Python — it just calls your `/predict` endpoint.

> **Project checklist.** ✅ split before preprocessing ✅ all preprocessing inside a `Pipeline` (no leakage) ✅ report the *right* metric for your problem (AUC/recall for imbalance, RMSE/R² for regression) ✅ save the whole pipeline, not just the bare model ✅ feed the deployed model raw data in the original schema.